# Exponential survival likelihood

Time starts at diagnosis and is measured in months; the event is death from any cause. Assume independent subjects, independent right censoring, and one constant common hazard. This is a teaching model, not an individual prediction or clinical recommendation.

For rate $\lambda>0$, $f(t)=\lambda e^{-\lambda t}$, $F(t)=1-e^{-\lambda t}$, $S(t)=e^{-\lambda t}$, and $h(t)=\lambda$. An event at $t_i$ contributes density $f(t_i)$; a right-censored subject known event-free through $t_i$ contributes $S(t_i)$, not $f(t_i)$. Thus $L(\lambda)=\prod_i[\lambda e^{-\lambda t_i}]^{d_i}[e^{-\lambda t_i}]^{1-d_i}=\lambda^D e^{-\lambda Y}$, where $D=\sum d_i$ and $Y=\sum t_i$. The log-likelihood is $D\log\lambda-\lambda Y$, its score is $D/\lambda-Y$, and its second derivative is $-D/\lambda^2$. For $D>0$, $\hat\lambda=D/Y$.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pytest
from lifelines import ExponentialFitter, KaplanMeierFitter, WeibullFitter
from scipy.optimize import minimize_scalar

from survivallab.data import fixed_fixture
from survivallab.stats import (
    fit_exponential,
    kaplan_meier,
    log_likelihood,
    rate_intervals,
)

rows = fixed_fixture()
times = np.array([row.duration_months for row in rows])
events = np.array([row.event_observed for row in rows])
fit = fit_exponential(times, events)
rate = fit.rate
assert rate is not None
naive_all_events = len(times) / times.sum()
event_only = events == 1
discard_censored = event_only.sum() / times[event_only].sum()
print(f"D={fit.events}; Y={fit.person_time}; lambda={rate:.10f}/month")
print(
    f"naive: all events={naive_all_events:.6f}; discard censored={discard_censored:.6f}"
)

In [ ]:
def survival(t):
    return math.exp(-rate * t)


def event_by(t):
    return -math.expm1(-rate * t)


print(
    {
        "mean_months": 1 / rate,
        "median_months": math.log(2) / rate,
        "S12": survival(12),
        "S24": survival(24),
        "S60": survival(60),
        "P(12<T<=24)": survival(12) - survival(24),
        "P(T>24 | T>12)": survival(12 + 12) / survival(12),
    }
)
print("P(event by 24 months):", event_by(24))

In [ ]:
result = minimize_scalar(
    lambda log_rate: -log_likelihood(math.exp(log_rate), times, events),
    bounds=(-10, 1),
    method="bounded",
)
optimizer_rate = math.exp(result.x)
lifelines_fit = ExponentialFitter().fit(times, event_observed=events)
lifelines_rate = 1 / lifelines_fit.lambda_
assert optimizer_rate == pytest.approx(rate, rel=1e-5)
assert lifelines_rate == pytest.approx(rate, rel=1e-5)
print(f"package={rate:.10f}; optimizer={optimizer_rate:.10f}")
print(f"lifelines rate=1/lambda_={lifelines_rate:.10f}")

In [ ]:
rate_grid = np.linspace(rate / 5, rate * 3, 300)
plt.plot(rate_grid, [log_likelihood(r, times, events) for r in rate_grid])
plt.axvline(rate, color="black", linestyle="--", label="D/Y")
plt.xlabel("Rate lambda (per month)")
plt.ylabel("Log-likelihood")
plt.legend()
plt.show()

horizon = np.linspace(0, 60, 300)
plt.plot(horizon, np.exp(-rate * horizon), label="Exponential fit")
km = kaplan_meier(times, events)
plt.step(
    [p["time_months"] for p in km],
    [p["survival"] for p in km],
    where="post",
    label="Kaplan-Meier",
)
plt.xlabel("Months since diagnosis")
plt.ylabel("Event-free probability")
plt.legend()
plt.show()
kmf = KaplanMeierFitter().fit(times, event_observed=events)
weibull = WeibullFitter().fit(times, event_observed=events)
plt.step(horizon, kmf.predict(horizon), where="post", label="Lifelines Kaplan-Meier")
plt.plot(
    horizon, weibull.survival_function_at_times(horizon), label="Weibull alternative"
)
plt.plot(horizon, np.exp(-rate * horizon), label="Exponential model")
plt.xlabel("Months since diagnosis")
plt.ylabel("Event-free probability")
plt.title("Diagnostic alternatives; exponential remains the fitted model")
plt.legend()
plt.show()
print("Asymptotic intervals:", rate_intervals(fit))

## Exercises (try before opening the answer cells)

1. Derive the estimator using the exponential scale $\theta=1/\lambda$.
2. Explain why duplicating every record leaves $D/Y$ unchanged but understates uncertainty if treated as independent people.
3. Explain why the memoryless assumption may not describe changing cancer hazards.

### Answers

1. The log-likelihood is $-D\log\theta-Y/\theta$; setting its derivative to zero gives $\hat\theta=Y/D$, so $\hat\lambda=1/\hat\theta=D/Y$. 2. Numerator and denominator double, but the apparent information doubles and standard errors shrink even though no independent evidence was collected. 3. Treatment, recurrence, frailty, and disease stage can change hazards over time, making a single rate a poor description and extrapolation risky.